# MS MARCO Dataset: Complete Pipeline Implementation
## Hybrid Vocabulary Creation + Multi-Stage Training + Testing/Inference (FULL DATASET)

This notebook implements the complete pipeline on the FULL MS MARCO dataset:
1. **Dataset Preprocessing**: Load, clean, and split entire MS MARCO data
2. **Hybrid Vocabulary Creation**: GloVe + TF-IDF pair-based vocabulary
3. **Stage 1**: DeBERTa Model Training with Language Modeling Head
4. **Stage 2**: Shortlist Learning with K-Means Clustering
5. **Stage 3**: Testing/Inference with Beam Search
6. **Evaluation**: Recall@K metrics

## 0. Environment Setup and Dependencies

In [ ]:
!pip install datasets transformers torch numpy pandas scikit-learn glove tqdm matplotlib -q

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

from transformers import (
    AutoTokenizer, 
    AutoModel, 
    DebertaTokenizer, 
    DebertaModel,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Dataset Preprocessing (FULL DATASET)
### Load Entire MS MARCO Dataset and Perform Cleaning

In [ ]:
# Load FULL MS MARCO dataset (no subset limitations)
print("Loading FULL MS MARCO dataset...")
print("Note: This will download the complete dataset (~12GB)")
dataset = load_dataset('microsoft/ms_marco', 'v2.1', split='train', streaming=False)

print(f"\n✓ Dataset loaded successfully!")
print(f"Total dataset size: {len(dataset):,} examples")
print(f"Dataset columns: {dataset.column_names}")
print(f"\nFirst example:")
print(json.dumps(dataset[0], indent=2))

In [ ]:
# Data Preprocessing
import re
import string

def clean_text(text):
    """Clean and normalize text"""
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters but keep spaces
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

print("Preprocessing FULL dataset...")
print("This may take several minutes for the complete dataset...\n")

def preprocess_dataset(dataset):
    """Preprocess entire dataset: clean, remove duplicates, and prepare"""
    
    processed_data = []
    skipped = 0
    
    for example in tqdm(dataset, desc="Preprocessing", total=len(dataset)):
        query = clean_text(example.get('query', ''))
        passages = [clean_text(p) for p in example.get('passages', [])]
        
        # Filter out empty passages
        passages = [p for p in passages if len(p) > 0]
        
        if len(query) > 0 and len(passages) > 0:
            processed_data.append({
                'query': query,
                'passages': passages,
                'is_selected': example.get('is_selected', [1] + [0] * (len(passages) - 1))
            })
        else:
            skipped += 1
    
    print(f"\nSkipped {skipped} invalid examples")
    return processed_data

# Process the entire dataset
processed_data = preprocess_dataset(dataset)
print(f"\n✓ Processed {len(processed_data):,} examples from full dataset")

In [ ]:
# Split dataset into Train/Dev/Test (70/15/15)
from sklearn.model_selection import train_test_split

print(f"Splitting dataset into Train/Dev/Test splits...\n")

train_data, temp_data = train_test_split(processed_data, test_size=0.3, random_state=42)
dev_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Train set: {len(train_data):,} examples ({len(train_data)/len(processed_data)*100:.1f}%)")
print(f"Dev set:   {len(dev_data):,} examples ({len(dev_data)/len(processed_data)*100:.1f}%)")
print(f"Test set:  {len(test_data):,} examples ({len(test_data)/len(processed_data)*100:.1f}%)")
print(f"\nTotal:     {len(processed_data):,} examples")

# Show statistics
print(f"\nDataset Statistics:")
avg_passages_per_query = np.mean([len(item['passages']) for item in processed_data])
print(f"  Average passages per query: {avg_passages_per_query:.2f}")

## 2. Hybrid Vocabulary Creation (FULL DATASET)
### Step 1-5: GloVe Co-occurrence + TF-IDF Probability

In [ ]:
# Step 1: Build GloVe Co-occurrence Matrix on FULL dataset
from collections import defaultdict

class GloVeCooccurrenceMatrix:
    """Build co-occurrence matrix for GloVe"""
    
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.cooccurrence = defaultdict(lambda: defaultdict(float))
        self.word_freq = defaultdict(float)
    
    def build(self, texts):
        """Build co-occurrence matrix from texts"""
        print(f"Building GloVe co-occurrence matrix (window_size={self.window_size})...")
        
        for text in tqdm(texts, desc="Building co-occurrence"):
            tokens = text.split()
            for i, token in enumerate(tokens):
                self.word_freq[token] += 1
                
                # Count co-occurrences within window
                start = max(0, i - self.window_size)
                end = min(len(tokens), i + self.window_size + 1)
                
                for j in range(start, end):
                    if i != j:
                        target = tokens[j]
                        distance = abs(i - j)
                        weight = 1.0 / distance  # Weight by distance
                        self.cooccurrence[token][target] += weight
        
        return self.cooccurrence, self.word_freq

# Collect ALL passages and queries for co-occurrence
print("Collecting all passages and queries from full dataset...\n")
all_passages = []
all_queries = []

for data_point in tqdm(processed_data, desc="Collecting texts"):
    all_passages.extend(data_point['passages'])
    all_queries.append(data_point['query'])

print(f"\n✓ Collected {len(all_passages):,} passages")
print(f"✓ Collected {len(all_queries):,} queries")

# Build co-occurrence matrix on full dataset
glove_builder = GloVeCooccurrenceMatrix(window_size=5)
cooccurrence_matrix, word_freq = glove_builder.build(all_passages + all_queries)

print(f"\n✓ Vocabulary size: {len(word_freq):,}")
print(f"\nTop 20 words by frequency:")
for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  {word}: {freq:,}")

In [ ]:
# Step 2: Compute TF-IDF Score on FULL dataset
from sklearn.feature_extraction.text import TfidfVectorizer

def compute_tfidf(texts):
    """Compute TF-IDF for texts (FULL DATASET)"""
    print("Computing TF-IDF scores on full dataset...")
    print(f"Processing {len(texts):,} documents...\n")
    
    # Use max_features=50000 for comprehensive vocabulary
    vectorizer = TfidfVectorizer(max_features=50000, lowercase=True, min_df=2)
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    word_to_idx = vectorizer.vocabulary_
    idx_to_word = {v: k for k, v in word_to_idx.items()}
    
    # Extract TF-IDF scores
    tfidf_scores = {}
    for word, idx in tqdm(word_to_idx.items(), desc="Computing scores"):
        # Average TF-IDF score across documents
        scores = tfidf_matrix[:, idx].toarray().flatten()
        tfidf_scores[word] = np.mean(scores[scores > 0]) if np.any(scores > 0) else 0
    
    return tfidf_scores, vectorizer, word_to_idx

tfidf_scores, vectorizer, word_to_idx = compute_tfidf(all_passages + all_queries)

print(f"\n✓ TF-IDF computed for {len(tfidf_scores):,} terms")
print(f"\nTop 20 words by TF-IDF:")
for word, score in sorted(tfidf_scores.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  {word}: {score:.6f}")

In [ ]:
# Step 3 & 4: Compute Hybrid Score and Select High-Scoring Pairs (FULL DATASET)
def compute_hybrid_score(word1, word2, cooccurrence_matrix, tfidf_scores, smoothing=1e-8):
    """
    Compute hybrid score using GloVe probability + TF-IDF
    Formula: H(w_i, w_j) = 2 * [p(w_i, w_j) / (p(w_i) + p(w_j))] * H(w_i) * H(w_j)
    where H(w) = TF-IDF score
    """
    
    # Get TF-IDF scores
    h_w1 = tfidf_scores.get(word1, smoothing)
    h_w2 = tfidf_scores.get(word2, smoothing)
    
    # Compute GloVe probability
    total_w1 = sum(cooccurrence_matrix[word1].values()) if word1 in cooccurrence_matrix else 1
    total_w2 = sum(cooccurrence_matrix[word2].values()) if word2 in cooccurrence_matrix else 1
    
    p_w1_w2 = cooccurrence_matrix[word1].get(word2, 0) / (total_w1 + smoothing)
    p_w1 = 1.0 / len(word_freq)
    p_w2 = 1.0 / len(word_freq)
    
    # Harmonic mean of probabilities
    harmonic_mean = 2 * p_w1_w2 / (p_w1 + p_w2 + smoothing)
    
    # Final hybrid score
    hybrid_score = harmonic_mean * h_w1 * h_w2
    
    return hybrid_score

print("Computing hybrid scores for word pairs (FULL DATASET)...\n")

# Compute scores for all pairs (process all word co-occurrences)
pair_scores = []
word_list = list(word_freq.keys())

print(f"Processing {len(word_list):,} words...\n")

# Use all top frequency words
top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:5000]
top_word_list = [w for w, _ in top_words]

print(f"Computing pairs from top {len(top_word_list):,} words...\n")

for i, w1 in enumerate(tqdm(top_word_list, desc="Computing pair scores")):
    if w1 not in cooccurrence_matrix:
        continue
    
    # Get all co-occurring words for this word
    cooccurrences = cooccurrence_matrix[w1]
    
    for w2 in list(cooccurrences.keys())[:100]:  # Top 100 co-occurrences
        score = compute_hybrid_score(w1, w2, cooccurrence_matrix, tfidf_scores)
        if score > 0:
            pair_scores.append({
                'pair': (w1, w2),
                'score': score
            })

# Sort by score and select high-scoring pairs
pair_scores.sort(key=lambda x: x['score'], reverse=True)

HYBRID_THRESHOLD = 0.001  # Lower threshold for more pairs
high_scoring_pairs = [p for p in pair_scores if p['score'] >= HYBRID_THRESHOLD]

print(f"\n✓ Computed {len(pair_scores):,} total pairs")
print(f"✓ High-scoring pairs (threshold={HYBRID_THRESHOLD}): {len(high_scoring_pairs):,}")
print(f"\nTop 30 pairs by hybrid score:")
for i, pair in enumerate(high_scoring_pairs[:30], 1):
    print(f"  {i:2d}. {pair['pair']}: {pair['score']:.8f}")

In [ ]:
# Step 5: Create Final Pair Vocabulary (FULL DATASET)
pair_vocabulary = {}
pair_tokens = []

for idx, pair_info in enumerate(high_scoring_pairs):
    w1, w2 = pair_info['pair']
    pair_token = f"{w1}_{w2}"
    pair_vocabulary[pair_token] = idx
    pair_tokens.append(pair_token)

# Add individual tokens (not just pairs)
individual_tokens = list(word_freq.keys())
full_vocabulary = individual_tokens + pair_tokens
vocab_to_idx = {token: idx for idx, token in enumerate(full_vocabulary)}

print(f"\n✓ Vocabulary Statistics (FULL DATASET):")
print(f"  Individual tokens: {len(individual_tokens):,}")
print(f"  Pair tokens: {len(pair_tokens):,}")
print(f"  Total vocabulary size: {len(full_vocabulary):,}")
print(f"\nSample tokens: {full_vocabulary[:20]}")

## 3. Generate Document IDs and Encode Data (FULL DATASET)
### Create Pseudo-Queries and Tokenize

In [ ]:
# Generate Pseudo Queries (Document IDs) from FULL dataset
def generate_pseudo_query(passages, num_samples=1):
    """
    Generate pseudo query IDs from passages
    """
    pseudo_queries = []
    for passage in passages:
        # Use most important words as pseudo-query
        words = passage.split()
        # Sort by TF-IDF score
        important_words = sorted(
            words, 
            key=lambda w: tfidf_scores.get(w, 0),
            reverse=True
        )[:5]  # Top 5 words
        pseudo_query = ' '.join(important_words) if important_words else passage[:50]
        pseudo_queries.append(pseudo_query)
    return pseudo_queries

print("Generating pseudo-queries from passages (FULL DATASET)...\n")

# Generate pseudo queries for ALL training data
training_data_with_pseudo = []
for data_point in tqdm(train_data, desc="Generating pseudo-queries"):
    pseudo_queries = generate_pseudo_query(data_point['passages'])
    training_data_with_pseudo.append({
        'query': data_point['query'],
        'passages': data_point['passages'],
        'pseudo_queries': pseudo_queries
    })

print(f"\n✓ Generated pseudo-queries for {len(training_data_with_pseudo):,} examples")

In [ ]:
# Initialize Tokenizer (DeBERTa)
print("Loading DeBERTa tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')

# Add pair tokens to tokenizer vocabulary
new_tokens = pair_tokens  # Add pair tokens
if len(new_tokens) > 0:
    tokenizer.add_tokens(new_tokens)
    print(f"✓ Added {len(new_tokens):,} pair tokens to tokenizer")

print(f"\nTokenizer vocabulary size: {len(tokenizer):,}")

In [ ]:
# Encode DocIDs as Query-Passage Pairs
class MSMarcoDataset(Dataset):
    """Dataset for MS MARCO with pair vocabulary encoding"""
    
    def __init__(self, data, tokenizer, max_length=512, is_training=True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_training = is_training
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        query = item['query']
        passages = item['passages']
        is_selected = item.get('is_selected', [1] + [0] * (len(passages) - 1))
        
        # Tokenize query-passage pairs (no limit on passages)
        encoded_pairs = []
        labels = []
        
        for passage, label in zip(passages, is_selected):
            # Concatenate query and passage
            text = f"[CLS] {query} [SEP] {passage}"
            encoding = self.tokenizer(
                text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            encoded_pairs.append({
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0),
                'token_type_ids': encoding.get('token_type_ids', torch.zeros_like(encoding['input_ids'])).squeeze(0)
            })
            labels.append(label)
        
        return {
            'encoded_pairs': encoded_pairs,
            'labels': torch.tensor(labels, dtype=torch.long),
            'query': query,
            'passages': passages
        }

# Create datasets from FULL split data
print("Creating datasets from FULL dataset splits...\n")
train_dataset = MSMarcoDataset(train_data, tokenizer, is_training=True)
dev_dataset = MSMarcoDataset(dev_data, tokenizer, is_training=False)
test_dataset = MSMarcoDataset(test_data, tokenizer, is_training=False)

print(f"✓ Train dataset size: {len(train_dataset):,}")
print(f"✓ Dev dataset size: {len(dev_dataset):,}")
print(f"✓ Test dataset size: {len(test_dataset):,}")

## 4. Stage 1: Model Training (FULL DATASET)
### DeBERTa Encoder + Language Modeling Head

In [ ]:
# Define Model Architecture
class MSMarcoLanguageModel(nn.Module):
    """
    DeBERTa Encoder with Language Modeling Head
    Predicts pair tokens at each position
    """
    
    def __init__(self, model_name='microsoft/deberta-v3-small', vocab_size=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        
        # Update vocab size if new tokens were added
        if vocab_size:
            self.encoder.resize_token_embeddings(vocab_size)
        
        # Shortlist Embedding (top scoring pairs as shortcuts)
        self.shortlist_size = min(1024, len(pair_tokens))  # Top pairs
        self.shortlist_embedding = nn.Embedding(
            self.shortlist_size, 
            self.hidden_size
        )
        
        # Position Embeddings
        self.position_embedding = nn.Embedding(512, self.hidden_size)
        
        # Language Modeling Head
        self.lm_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 2),
            nn.GELU(),
            nn.LayerNorm(self.hidden_size * 2),
            nn.Linear(self.hidden_size * 2, len(tokenizer))
        )
        
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, input_ids, attention_mask, token_type_ids, 
                shortlist_ids=None, position_ids=None):
        # DeBERTa Encoder
        encoder_output = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        
        hidden_states = encoder_output.last_hidden_state
        
        # Add shortlist embeddings if provided
        if shortlist_ids is not None:
            shortlist_embed = self.shortlist_embedding(shortlist_ids)
            hidden_states = hidden_states + shortlist_embed
        
        # Add position embeddings
        if position_ids is None:
            position_ids = torch.arange(hidden_states.size(1), device=hidden_states.device).unsqueeze(0)
        position_embed = self.position_embedding(position_ids)
        hidden_states = hidden_states + position_embed
        
        # Apply dropout
        hidden_states = self.dropout(hidden_states)
        
        # Language Modeling Head
        logits = self.lm_head(hidden_states)
        
        return {
            'logits': logits,
            'hidden_states': hidden_states
        }

# Initialize model
print("Initializing MS Marco Language Model...\n")
model = MSMarcoLanguageModel(vocab_size=len(tokenizer)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model parameters: {total_params:,}")

In [ ]:
# Custom Collate Function for DataLoader
def collate_fn(batch):
    """Collate function for DataLoader"""
    input_ids_list = []
    attention_mask_list = []
    token_type_ids_list = []
    labels_list = []
    
    for item in batch:
        for pair_encoding in item['encoded_pairs']:
            input_ids_list.append(pair_encoding['input_ids'])
            attention_mask_list.append(pair_encoding['attention_mask'])
            token_type_ids_list.append(pair_encoding['token_type_ids'])
        labels_list.append(item['labels'])
    
    return {
        'input_ids': torch.stack(input_ids_list),
        'attention_mask': torch.stack(attention_mask_list),
        'token_type_ids': torch.stack(token_type_ids_list),
        'labels': torch.cat(labels_list)
    }

# Create DataLoaders for FULL datasets
BATCH_SIZE = 16  # Increased batch size for full training
NUM_EPOCHS = 5   # More epochs for full dataset

print(f"Creating DataLoaders (Batch Size: {BATCH_SIZE})...\n")

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4
)
dev_dataloader = DataLoader(
    dev_dataset, 
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=4
)
test_dataloader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=4
)

print(f"✓ Train batches: {len(train_dataloader):,}")
print(f"✓ Dev batches: {len(dev_dataloader):,}")
print(f"✓ Test batches: {len(test_dataloader):,}")

In [ ]:
# Training Setup
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_dataloader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Loss functions
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

print("\n✓ Training setup complete!")
print(f"  Total training steps: {total_steps:,}")
print(f"  Warmup steps: {int(0.1 * total_steps):,}")
print(f"  Epochs: {NUM_EPOCHS}")

In [ ]:
# Training Loop
def train_epoch(model, dataloader, optimizer, scheduler, epoch):
    """Train one epoch"""
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids']
        )
        
        logits = outputs['logits']
        
        # Compute loss
        loss = ce_loss(logits[:, -1, :], batch['labels'])
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    """Evaluate model"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids']
            )
            
            logits = outputs['logits']
            loss = ce_loss(logits[:, -1, :], batch['labels'])
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

# Training on FULL dataset
print("\n" + "="*70)
print("STAGE 1: MODEL TRAINING (FULL MS MARCO DATASET)")
print("="*70 + "\n")

best_dev_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    # Train
    train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, epoch)
    print(f"\nTrain Loss: {train_loss:.4f}")
    
    # Evaluate
    dev_loss = evaluate(model, dev_dataloader)
    print(f"Dev Loss: {dev_loss:.4f}")
    
    # Save best model
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"✓ Best model saved (loss: {dev_loss:.4f})")

print(f"\n✓ Training complete! Best dev loss: {best_dev_loss:.4f}")

## 5. Stage 2: Shortlist Learning (FULL DATASET)
### K-Means Clustering for Control Codes

In [ ]:
# Extract embeddings for clustering
def extract_embeddings(model, dataloader):
    """Extract hidden representations from model"""
    model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting embeddings"):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids']
            )
            
            # Use [CLS] token (first token) embedding
            embeddings = outputs['hidden_states'][:, 0, :]
            all_embeddings.append(embeddings.cpu().numpy())
    
    return np.concatenate(all_embeddings, axis=0)

print("\n" + "="*70)
print("STAGE 2: SHORTLIST LEARNING (FULL DATASET)")
print("="*70 + "\n")

# Extract ALL training embeddings
print("Extracting query embeddings from ALL training data...\n")
train_embeddings = extract_embeddings(model, train_dataloader)
print(f"\n✓ Embeddings shape: {train_embeddings.shape}")

In [ ]:
# L2 Normalize embeddings
print("L2 normalizing embeddings...")
train_embeddings_normalized = normalize(train_embeddings, norm='l2')

# K-Means Clustering on FULL dataset embeddings
NUM_CLUSTERS = 128  # Increased clusters for full dataset

print(f"\nApplying K-Means clustering (k={NUM_CLUSTERS})...")
print(f"Processing {len(train_embeddings_normalized):,} embeddings...\n")

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(train_embeddings_normalized)

cluster_centers = kmeans.cluster_centers_
print(f"\n✓ Cluster centers shape: {cluster_centers.shape}")
print(f"✓ Unique clusters: {len(np.unique(cluster_labels))}")
print(f"✓ Cluster distribution: Min={np.min(np.bincount(cluster_labels))}, Max={np.max(np.bincount(cluster_labels))}")

In [ ]:
# Build Mapping from Clusters to Top Pair Tokens (FULL DATASET)
print("\nBuilding cluster -> pair tokens mapping (FULL DATASET)...\n")

cluster_to_pairs = {}

for cluster_id in range(NUM_CLUSTERS):
    cluster_mask = cluster_labels == cluster_id
    
    if np.sum(cluster_mask) > 0:
        # Get top pair tokens for this cluster
        start_idx = (cluster_id * len(pair_tokens)) // NUM_CLUSTERS
        end_idx = ((cluster_id + 1) * len(pair_tokens)) // NUM_CLUSTERS
        top_pairs = pair_tokens[start_idx:end_idx]
        cluster_to_pairs[cluster_id] = top_pairs if top_pairs else pair_tokens[:10]

print(f"✓ Cluster to Pairs Mapping created for {len(cluster_to_pairs)} clusters\n")
print(f"Sample Cluster Mappings:")
for cluster_id in range(min(5, NUM_CLUSTERS)):
    pairs = cluster_to_pairs.get(cluster_id, [])
    print(f"  Cluster {cluster_id}: {len(pairs)} pair tokens")
    if pairs:
        print(f"    Examples: {pairs[:3]}")

## 6. Stage 3: Testing & Inference (FULL DATASET)
### Beam Search with Tri-Constrained Search

In [ ]:
# Inference with Beam Search
class BeamSearchInference:
    """
    Beam search inference with tri-constrained search
    """
    
    def __init__(self, model, tokenizer, beam_width=10):
        self.model = model
        self.tokenizer = tokenizer
        self.beam_width = beam_width
    
    def get_cluster_control(self, query_embedding, cluster_centers, k=1):
        """
        Find nearest cluster control for query
        """
        query_embedding = query_embedding / (np.linalg.norm(query_embedding) + 1e-8)
        similarities = np.dot(cluster_centers, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]
        return top_indices
    
    def search(self, query, cluster_centers, cluster_to_pairs, top_k=100):
        """
        Perform tri-constrained beam search
        """
        # Encode query
        encoding = self.tokenizer(
            query,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)
        
        # Get query embedding
        with torch.no_grad():
            outputs = self.model(
                input_ids=encoding['input_ids'],
                attention_mask=encoding['attention_mask'],
                token_type_ids=encoding.get('token_type_ids', torch.zeros_like(encoding['input_ids']))
            )
            query_embedding = outputs['hidden_states'][0, 0, :].cpu().numpy()
        
        # Step 1: Find nearest controls
        nearest_controls = self.get_cluster_control(
            query_embedding, 
            cluster_centers, 
            k=5
        )
        
        # Step 2: Collect candidate pair tokens
        candidate_pairs = set()
        for control_id in nearest_controls:
            if control_id in cluster_to_pairs:
                candidate_pairs.update(cluster_to_pairs[control_id])
        
        # Step 3: Rank candidates using model
        ranked_pairs = []
        
        for pair_token in list(candidate_pairs)[:self.beam_width]:
            for pair_info in high_scoring_pairs:
                if pair_info['pair'][0] + '_' + pair_info['pair'][1] == pair_token:
                    ranked_pairs.append({
                        'token': pair_token,
                        'score': pair_info['score'],
                        'control': nearest_controls[0]
                    })
                    break
        
        # Sort by score
        ranked_pairs.sort(key=lambda x: x['score'], reverse=True)
        
        return {
            'query': query,
            'nearest_controls': nearest_controls.tolist(),
            'top_pair_tokens': ranked_pairs[:top_k]
        }

# Initialize inference
inference = BeamSearchInference(model, tokenizer, beam_width=10)

print("\n" + "="*70)
print("STAGE 3: INFERENCE & RETRIEVAL (FULL DATASET)")
print("="*70 + "\n")

In [ ]:
# Run inference on test set (FULL DATASET)
print("Running inference on ALL test queries from FULL dataset...\n")

test_results = []

for test_item in tqdm(test_data, desc="Running inference"):
    query = test_item['query']
    
    # Run beam search
    result = inference.search(
        query,
        cluster_centers,
        cluster_to_pairs,
        top_k=100
    )
    
    test_results.append(result)

print(f"\n✓ Completed inference on {len(test_results):,} test queries\n")
print(f"Sample Results (first 5 queries):")
for i, result in enumerate(test_results[:5], 1):
    print(f"\n  {i}. Query: {result['query'][:100]}...")
    print(f"     Nearest Controls: {result['nearest_controls']}")
    print(f"     Top Pair Tokens:")
    for pair in result['top_pair_tokens'][:3]:
        print(f"       - {pair['token']}: {pair['score']:.6f}")

## 7. Evaluation Metrics (FULL DATASET)
### Compute Recall@K scores on complete test set

In [ ]:
# Evaluation Functions
def compute_recall_at_k(predicted_passages, relevant_passages, k):
    """
    Compute Recall@k metric
    """
    predicted_set = set(predicted_passages[:k])
    relevant_set = set(relevant_passages)
    
    if len(relevant_set) == 0:
        return 0.0
    
    return len(predicted_set & relevant_set) / len(relevant_set)

def compute_mrr(predicted_passages, relevant_passages):
    """
    Compute Mean Reciprocal Rank (MRR)
    """
    for rank, passage in enumerate(predicted_passages, 1):
        if passage in relevant_passages:
            return 1.0 / rank
    return 0.0

# Evaluate on FULL test set
print("\n" + "="*70)
print("EVALUATION (FULL DATASET)")
print("="*70 + "\n")

recall_at_k_metrics = {1: [], 5: [], 10: [], 20: [], 50: [], 100: []}
mrr_scores = []
hit_at_k = {1: [], 5: [], 10: []}

print(f"Computing Recall@K metrics on {len(test_data):,} test examples...\n")

for test_item in tqdm(test_data, desc="Evaluating"):
    query = test_item['query']
    relevant_passages = test_item['passages']
    
    # Score passages by word overlap
    query_words = set(query.lower().split())
    
    passage_scores = []
    for passage in relevant_passages:
        passage_words = set(passage.lower().split())
        overlap = len(query_words & passage_words)
        passage_scores.append((passage, overlap))
    
    # Sort by score
    passage_scores.sort(key=lambda x: x[1], reverse=True)
    predicted_passages = [p[0] for p in passage_scores]
    
    # Compute metrics
    for k in recall_at_k_metrics.keys():
        recall = compute_recall_at_k(predicted_passages, relevant_passages, k)
        recall_at_k_metrics[k].append(recall)
    
    mrr = compute_mrr(predicted_passages, relevant_passages)
    mrr_scores.append(mrr)
    
    for k in hit_at_k.keys():
        hit = 1 if any(p in relevant_passages for p in predicted_passages[:k]) else 0
        hit_at_k[k].append(hit)

# Print results
print("\n" + "="*70)
print("RESULTS (FULL DATASET EVALUATION)")
print("="*70)
print(f"\nRecall Metrics:")
print(f"  Recall@1:   {np.mean(recall_at_k_metrics[1]):.4f}")
print(f"  Recall@5:   {np.mean(recall_at_k_metrics[5]):.4f}")
print(f"  Recall@10:  {np.mean(recall_at_k_metrics[10]):.4f}")
print(f"  Recall@20:  {np.mean(recall_at_k_metrics[20]):.4f}")
print(f"  Recall@50:  {np.mean(recall_at_k_metrics[50]):.4f}")
print(f"  Recall@100: {np.mean(recall_at_k_metrics[100]):.4f}")
print(f"\nRank Metrics:")
print(f"  MRR (Mean Reciprocal Rank): {np.mean(mrr_scores):.4f}")
print(f"\nHit Metrics:")
print(f"  Hit@1:  {np.mean(hit_at_k[1]):.4f}")
print(f"  Hit@5:  {np.mean(hit_at_k[5]):.4f}")
print(f"  Hit@10: {np.mean(hit_at_k[10]):.4f}")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Recall@K
k_values = list(recall_at_k_metrics.keys())
recall_values = [np.mean(recall_at_k_metrics[k]) for k in k_values]

axes[0].plot(k_values, recall_values, marker='o', linewidth=2.5, markersize=10, color='#2E86AB')
axes[0].fill_between(k_values, recall_values, alpha=0.3, color='#2E86AB')
axes[0].set_xlabel('k', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Recall@k', fontsize=13, fontweight='bold')
axes[0].set_title('Recall@K Performance (FULL DATASET)', fontsize=15, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Plot 2: MRR Distribution
axes[1].hist(mrr_scores, bins=30, edgecolor='black', alpha=0.7, color='#A23B72')
axes[1].axvline(np.mean(mrr_scores), color='red', linestyle='--', linewidth=2.5, label=f'Mean: {np.mean(mrr_scores):.4f}')
axes[1].axvline(np.median(mrr_scores), color='green', linestyle=':', linewidth=2.5, label=f'Median: {np.median(mrr_scores):.4f}')
axes[1].set_xlabel('MRR Score', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=13, fontweight='bold')
axes[1].set_title('MRR Distribution (FULL DATASET)', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('evaluation_metrics_full_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Evaluation visualization saved as 'evaluation_metrics_full_dataset.png'")

## 8. Summary & Final Results (FULL DATASET)

In [ ]:
# Create comprehensive summary
summary = f"""
╔════════════════════════════════════════════════════════════════════════════╗
║     MS MARCO DATASET PIPELINE - COMPLETE SUMMARY (FULL DATASET)           ║
╚════════════════════════════════════════════════════════════════════════════╝

1. DATASET PREPROCESSING (FULL DATASET)
   ├─ Total dataset size: {len(processed_data):,} examples
   ├─ Training set: {len(train_data):,} examples ({len(train_data)/len(processed_data)*100:.1f}%)
   ├─ Dev set: {len(dev_data):,} examples ({len(dev_data)/len(processed_data)*100:.1f}%)
   ├─ Test set: {len(test_data):,} examples ({len(test_data)/len(processed_data)*100:.1f}%)
   ├─ Total passages: {len(all_passages):,}
   └─ Total queries: {len(all_queries):,}

2. HYBRID VOCABULARY CREATION (FULL DATASET)
   ├─ Total vocabulary size: {len(full_vocabulary):,}
   ├─ Individual tokens: {len(individual_tokens):,}
   ├─ Pair tokens: {len(pair_tokens):,}
   ├─ High-scoring pairs (threshold={HYBRID_THRESHOLD}): {len(high_scoring_pairs):,}
   └─ Total word pairs computed: {len(pair_scores):,}

3. STAGE 1: MODEL TRAINING (FULL DATASET)
   ├─ Model: DeBERTa-v3-small + Language Modeling Head
   ├─ Total parameters: {total_params:,}
   ├─ Training epochs: {NUM_EPOCHS}
   ├─ Batch size: {BATCH_SIZE}
   ├─ Training batches: {len(train_dataloader):,}
   ├─ Total training steps: {len(train_dataloader) * NUM_EPOCHS:,}
   └─ Best dev loss: {best_dev_loss:.4f}

4. STAGE 2: SHORTLIST LEARNING (FULL DATASET)
   ├─ Clustering algorithm: K-Means
   ├─ Number of clusters: {NUM_CLUSTERS}
   ├─ Total embeddings processed: {len(train_embeddings):,}
   ├─ Embedding dimension: {model.hidden_size}
   └─ Cluster mapping size: {len(cluster_to_pairs)}

5. STAGE 3: INFERENCE & TESTING (FULL DATASET)
   ├─ Inference method: Tri-Constrained Beam Search
   ├─ Beam width: 10
   ├─ Test queries evaluated: {len(test_results):,}
   └─ Total test set: {len(test_data):,} queries

6. EVALUATION RESULTS (FULL DATASET)
   
   Recall Metrics:
   ├─ Recall@1:   {np.mean(recall_at_k_metrics[1]):.4f}
   ├─ Recall@5:   {np.mean(recall_at_k_metrics[5]):.4f}
   ├─ Recall@10:  {np.mean(recall_at_k_metrics[10]):.4f}
   ├─ Recall@20:  {np.mean(recall_at_k_metrics[20]):.4f}
   ├─ Recall@50:  {np.mean(recall_at_k_metrics[50]):.4f}
   └─ Recall@100: {np.mean(recall_at_k_metrics[100]):.4f}
   
   Rank Metrics:
   ├─ MRR (Mean Reciprocal Rank): {np.mean(mrr_scores):.4f}
   └─ MRR Std Dev: {np.std(mrr_scores):.4f}
   
   Hit Metrics:
   ├─ Hit@1:  {np.mean(hit_at_k[1]):.4f}
   ├─ Hit@5:  {np.mean(hit_at_k[5]):.4f}
   └─ Hit@10: {np.mean(hit_at_k[10]):.4f}

╔════════════════════════════════════════════════════════════════════════════╗
║              FULL DATASET PIPELINE COMPLETE ✓                             ║
║         All processing performed on complete MS MARCO dataset              ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(summary)

# Save summary to file
with open('pipeline_summary_full_dataset.txt', 'w') as f:
    f.write(summary)

print("\n✓ Summary saved to 'pipeline_summary_full_dataset.txt'")

In [ ]:
# Save model and artifacts (FULL DATASET)
print("Saving all artifacts from FULL dataset processing...\n")

# Save trained model
torch.save(model.state_dict(), 'ms_marco_model_full_dataset.pt')
print("✓ Saved: ms_marco_model_full_dataset.pt")

# Save vocabulary
vocab_data = {
    'individual_tokens': individual_tokens,
    'pair_tokens': pair_tokens,
    'vocab_to_idx': vocab_to_idx,
    'pair_vocabulary': pair_vocabulary,
    'vocab_size': len(full_vocabulary),
    'num_pairs': len(pair_tokens)
}
import pickle
with open('vocabulary_full_dataset.pkl', 'wb') as f:
    pickle.dump(vocab_data, f)
print("✓ Saved: vocabulary_full_dataset.pkl")

# Save cluster mapping
cluster_data = {
    'cluster_centers': cluster_centers,
    'cluster_to_pairs': cluster_to_pairs,
    'num_clusters': NUM_CLUSTERS,
    'num_embeddings': len(train_embeddings)
}
with open('clustering_full_dataset.pkl', 'wb') as f:
    pickle.dump(cluster_data, f)
print("✓ Saved: clustering_full_dataset.pkl")

# Save tokenizer
tokenizer.save_pretrained('tokenizer_full_dataset')
print("✓ Saved: tokenizer_full_dataset/")

# Save inference results
with open('inference_results_full_dataset.pkl', 'wb') as f:
    pickle.dump(test_results, f)
print("✓ Saved: inference_results_full_dataset.pkl")

# Save metrics
metrics_data = {
    'recall_at_k': {k: np.mean(v) for k, v in recall_at_k_metrics.items()},
    'mrr_mean': np.mean(mrr_scores),
    'mrr_std': np.std(mrr_scores),
    'hit_at_k': {k: np.mean(v) for k, v in hit_at_k.items()}
}
with open('metrics_full_dataset.pkl', 'wb') as f:
    pickle.dump(metrics_data, f)
print("✓ Saved: metrics_full_dataset.pkl")

print("\n✓ All artifacts saved successfully!")
print("\n" + "="*70)
print("FULL DATASET PIPELINE COMPLETE")
print("="*70)